In [2]:
%pip -q install qdrant-client sentence-transformers google-generativeai python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: C:\Users\yara2\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:

import os
from dotenv import load_dotenv
load_dotenv()

import google.generativeai as genai
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", None)
COLLECTION = os.getenv("QDRANT_COLLECTION", "inspira_chunks_v1")

EMBED_MODEL_NAME = os.getenv("EMBED_MODEL_NAME", "paraphrase-multilingual-MiniLM-L12-v2")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "models/gemini-2.5-flash")

if not GEMINI_API_KEY:
    raise RuntimeError("GEMINI_API_KEY is missing in .env")

qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
qdrant.get_collections()
print("✅ Qdrant connected")

embedder = SentenceTransformer(EMBED_MODEL_NAME)

genai.configure(api_key=GEMINI_API_KEY)
llm = genai.GenerativeModel(GEMINI_MODEL)

def embed(text: str):
    return embedder.encode(text, normalize_embeddings=True).tolist()


D:\pip_cache\ipykernel_25952\1901164092.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai
C:\Users\yara2\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\transformers\utils\hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


✅ Qdrant connected


In [4]:
qdrant = QdrantClient(url="http://localhost:6333")
print(qdrant.get_collections())


collections=[CollectionDescription(name='inspira_chunks_v1')]


In [5]:
import google.generativeai as genai

models = list(genai.list_models())
for m in models:
    if "generateContent" in m.supported_generation_methods:
        print(m.name)


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025
models/antigravity-preview-05-2026
models/

In [6]:
# =========================
# Cell 2 — Structure + Intents + Draft Context Builder (FROM context_preparation.ipynb)
# =========================

TECH_OFFER_STRUCTURE = [
    {"key": "intro",              "title": "1) مقدمة وفهم عام للمشروع",          "required": True},
    {"key": "scope",              "title": "2) فهم نطاق العمل والمتطلبات",        "required": True},
    {"key": "methodology",        "title": "3) منهجية التنفيذ وآلية العمل",       "required": True},
    {"key": "timeline",           "title": "4) خطة العمل والجدول الزمني",          "required": True},
    {"key": "deliverables",       "title": "5) المخرجات والتسليمات",              "required": True},

    {"key": "team",               "title": "6) فريق العمل والحوكمة",              "required": False},
    {"key": "quality",            "title": "7) ضمان الجودة ومعايير القبول",        "required": False},
    {"key": "risk",               "title": "8) إدارة المخاطر",                    "required": False},
    {"key": "assumptions",        "title": "9) الافتراضات والقيود",               "required": False},
    
    {"key": "compliance",         "title": "10) الالتزام والمتطلبات النظامية",     "required": True},
]


TECH_OFFER_INTENTS = {
    "intro": {
        "rfp_intent": "استخرج وصف المشروع وأهدافه وخلفيته وسبب تنفيذ المشروع. ابحث عن أي فقرة تعطي صورة عامة عن المشروع.",
        "tp_intent":  "اجلب أسلوب كتابة مقدمة عرض فني رسمي: تعريف بالمشروع، الهدف، والنطاق العام."
    },
    "scope": {
        "rfp_intent": "استخرج نطاق العمل: المهام، الأنشطة، المتطلبات الفنية، وما المطلوب تنفيذه. ركّز على ما يجب تسليمه.",
        "tp_intent":  "اجلب أسلوب صياغة قسم فهم نطاق العمل والمتطلبات وكيف يتم عرضها بشكل منظم."
    },
    "methodology": {
        "rfp_intent": "استخرج أي توجيهات أو متطلبات تخص طريقة التنفيذ أو إدارة المشروع أو مراحل العمل.",
        "tp_intent":  "اجلب أسلوب كتابة المنهجية: مراحل، آلية عمل، إدارة مشروع، تواصل، تقارير."
    },
    "timeline": {
        "rfp_intent": "استخرج المدة الزمنية، الجدول الزمني، المراحل، التواريخ أو الأرباع/الأشهر، وأي متطلبات تسليم زمنية.",
        "tp_intent":  "اجلب أسلوب كتابة خطة عمل وجدول زمني في عرض فني (مراحل + مدة + تسليمات)."
    },
    "deliverables": {
        "rfp_intent": "استخرج قائمة المخرجات: وثائق، تقارير، أنظمة، تدريب، تسليمات نهائية ومعايير الاستلام.",
        "tp_intent":  "اجلب أسلوب كتابة قسم المخرجات والتسليمات وكيف يتم سردها بشكل احترافي."
    },
    "compliance": {
        "rfp_intent": "استخرج أي التزامات نظامية أو لائحية أو شروط امتثال أو معايير يجب الالتزام بها.",
        "tp_intent":  "اجلب أسلوب صياغة فقرة الالتزام والامتثال بشكل رسمي وواضح."
    }
}


import re
from typing import Dict, List, Any, Optional
from qdrant_client.http import models as qm


def generate_tech_offer_draft_with_intents(
    pair_id: int,
    project_title: str,
    structure: List[Dict[str, Any]],
    client,
    collection_name: str,
    embed_fn,
    intents: Optional[Dict[str, Dict[str, str]]] = None,
    include_tp_support: bool = True,
    top_k: int = 8,
    min_score: float = 0.35,
    max_chars_rfp: int = 2500,
    max_chars_tp: int = 1200,
    min_chunk_chars: int = 80,
    dedup: bool = True,
    include_evidence_headers: bool = False,
    scope_text: str = "",
    timeline_text: str = "",
) -> str:
    """
    Generate a technical-offer draft (Markdown) using intent-based retrieval.

    - Uses structure (required=True only)
    - Retrieves from RFP as primary evidence
    - Optionally retrieves from TP as style support
    - Uses intents (descriptions) rather than exact keyword matching
    """

    # -----------------------------
    # 1) Default intents (if not provided)
    # -----------------------------
    DEFAULT_INTENTS = {
        "intro": {
            "rfp": "استخرج وصف المشروع وأهدافه وخلفيته وسبب تنفيذ المشروع. ركّز على أي فقرة تعطي صورة عامة عن المشروع.",
            "tp":  "اجلب أسلوب كتابة مقدمة عرض فني رسمي: تعريف بالمشروع، الهدف، والنطاق العام."
        },
        "scope": {
            "rfp": "استخرج نطاق العمل: المهام، الأنشطة، المتطلبات الفنية، وما المطلوب تنفيذه. ركّز على المطلوب تنفيذه وما يجب تسليمه.",
            "tp":  "اجلب أسلوب صياغة قسم فهم نطاق العمل والمتطلبات وكيف يتم عرضها بشكل منظم."
        },
        "methodology": {
            "rfp": "استخرج أي توجيهات أو متطلبات تخص طريقة التنفيذ أو إدارة المشروع أو مراحل العمل أو آلية التواصل والتقارير.",
            "tp":  "اجلب أسلوب كتابة المنهجية: مراحل، آلية عمل، إدارة مشروع، تواصل، تقارير."
        },
        "timeline": {
            "rfp": "استخرج المدة الزمنية، الجدول الزمني، المراحل، التواريخ أو الأرباع/الأشهر، وأي متطلبات تسليم زمنية.",
            "tp":  "اجلب أسلوب كتابة خطة عمل وجدول زمني في عرض فني (مراحل + مدة + تسليمات)."
        },
        "deliverables": {
            "rfp": "استخرج قائمة المخرجات: وثائق، تقارير، أنظمة، تدريب، تسليمات نهائية ومعايير الاستلام.",
            "tp":  "اجلب أسلوب كتابة قسم المخرجات والتسليمات وكيف يتم سردها بشكل احترافي."
        },
        "compliance": {
            "rfp": "استخرج أي التزامات نظامية أو لائحية أو شروط امتثال أو معايير يجب الالتزام بها.",
            "tp":  "اجلب أسلوب صياغة فقرة الالتزام والامتثال بشكل رسمي وواضح."
        }
    }

    # Normalize intents format
    intents_norm = {}
    if intents:
        for k, v in intents.items():
            # supports both {"rfp_intent": ..., "tp_intent": ...} and {"rfp": ..., "tp": ...}
            intents_norm[k] = {
                "rfp": v.get("rfp") or v.get("rfp_intent") or DEFAULT_INTENTS.get(k, {}).get("rfp", ""),
                "tp":  v.get("tp")  or v.get("tp_intent")  or DEFAULT_INTENTS.get(k, {}).get("tp",  "")
            }
    else:
        intents_norm = DEFAULT_INTENTS

    # -----------------------------
    # 2) Helpers
    # -----------------------------
    def _clean_text(s: str) -> str:
        s = (s or "").strip()
        s = re.sub(r"\s+", " ", s)
        return s

    def _intent_to_query(key: str, doc_type: str) -> str:
        # doc_type: "rfp" or "tp"
        base = intents_norm.get(key, {}).get(doc_type, "")
        # Add a little scaffolding to help retrieval
        return f"{project_title}\n{base}".strip()

    def _retrieve(doc_type: str, query: str, limit: int):
        # Create vector
        vec = embed_fn(query)

        # Filter by doc_type only
        flt = qm.Filter(
            must=[
                qm.FieldCondition(
                    key="doc_type",
                    match=qm.MatchValue(value=doc_type),
                ),
            ]
        )
        hits = client.search(
            collection_name=collection_name,
            query_vector=vec,
            query_filter=flt,
            limit=limit,
            score_threshold=min_score,
            with_payload=True
        )
        return hits

    def _build_context(hits, max_chars: int) -> str:
        if not hits:
            return ""

        seen = set()
        parts = []
        total = 0

        # higher score first
        hits_sorted = sorted(hits, key=lambda x: x.score, reverse=True)

        for h in hits_sorted:
            payload = h.payload or {}
            txt = payload.get("text", "")
            txt = _clean_text(txt)

            if len(txt) < min_chunk_chars:
                continue

            if dedup:
                sig = (payload.get("chunk_id"), txt[:80])
                if sig in seen:
                    continue
                seen.add(sig)

            if include_evidence_headers:
                block = f"- (score={h.score:.3f}) {txt}"
            else:
                block = txt

            if total + len(block) > max_chars:
                block = block[: max(0, max_chars - total)]
            if block:
                parts.append(block)
                total += len(block)
            if total >= max_chars:
                break

        return "\n".join(parts).strip()

    def _user_injection(key: str) -> str:
        # Inject user-provided scope/timeline into relevant sections
        out = ""
        if key == "scope" and scope_text.strip():
            out += f"**مدخلات المستخدم (نطاق العمل):** {scope_text.strip()}\n\n"
        if key == "timeline" and timeline_text.strip():
            out += f"**مدخلات المستخدم (الجدول الزمني):** {timeline_text.strip()}\n\n"
        return out

    # -----------------------------
    # 3) Build Markdown Draft
    # -----------------------------
    md = []
    md.append("# العرض الفني\n")
    md.append(f"**اسم المشروع:** {project_title}\n")
    md.append(f"**رقم المشروع (pair_id):** {pair_id}\n")

    for sec in structure:
        # Skip optional sections
        if not sec.get("required", False):
            continue

        key = sec["key"]
        title = sec["title"]

        # ---- Primary RFP
        q_rfp = _intent_to_query(key, "rfp")
        rfp_hits = _retrieve("rfp", q_rfp, limit=top_k)
        rfp_ctx = _build_context(rfp_hits, max_chars=max_chars_rfp)

        # ---- TP support (optional)
        tp_ctx = ""
        if include_tp_support:
            q_tp = _intent_to_query(key, "tp")
            tp_hits = _retrieve("tp", q_tp, limit=max(3, top_k // 2))
            tp_ctx = _build_context(tp_hits, max_chars=max_chars_tp)

        # Fallback
        if not rfp_ctx:
            rfp_ctx = "(لم يتم العثور على مقاطع كافية من الكراسة لهذا القسم. جرّبي رفع top_k أو خفض min_score أو تأكدي من pair_id.)"

        md.append(f"\n## {title}\n")
        md.append(_user_injection(key) + rfp_ctx)

        if tp_ctx:
            md.append("\n\n**مرجع أسلوبي (TP):**\n")
            md.append(tp_ctx)

    return "\n".join(md).strip()


In [7]:
SYSTEM_PROMPT_FINAL = """
أنت كاتب عروض فنية حكومية سعودية (Technical Proposal) بصياغة رسمية مهنية.

## مصادر المعلومات (مصنّفة بوضوح):
1) RFP_EVIDENCE (مصدر حقائق):
   - مقتطفات من كراسة المنافسة/طلب تقديم العروض.
   - يُسمح لك باستخلاص الحقائق والمتطلبات منه فقط.

2) USER_INPUTS (مصدر حقائق إضافي):
   - نطاق عمل وشروط خاصة أدخلها المستخدم.
   - تعتبر حقائق ما لم تتعارض مع RFP_EVIDENCE.

3) TP_STYLE (أسلوب فقط وليس حقائق):
   - مقتطفات من عروض فنية سابقة.
   - يُستخدم فقط لتقليد النبرة وطريقة العرض، وممنوع استخراج متطلبات/أرقام/أسماء/تفاصيل منه.

## قواعد صارمة (غير قابلة للتجاوز):
1) ممنوع النسخ:
   - لا تنقل جملًا أو فقرات أو أرقامًا أو أسماء من RFP_EVIDENCE أو TP_STYLE حرفيًا.
   - أعد الصياغة دائمًا بأسلوب جديد.

2) ممنوع الاختلاق:
   - أي معلومة غير موجودة في RFP_EVIDENCE أو USER_INPUTS يجب أن تُكتب بصيغة:
     "يتطلب تأكيد من الجهة" أو "سيتم تحديده لاحقًا وفق وثائق المنافسة" دون تخمين.

3) التزم بالهيكل:
   - اكتب العرض الفني بالكامل وفق العناوين وترتيبها كما تظهر في TECH_OFFER_STRUCTURE ضمن السياق.
   - لا تضف أقسامًا جديدة ولا تغيّر العناوين.

4) أسلوب حكومي تنفيذي:
   - لغة واضحة، محددة، قابلة للتدقيق، بدون تسويق أو مبالغة.
   - استخدم نقاط/قوائم عند تعداد الأعمال أو المخرجات.
   - ركّز على "كيف سيتم التنفيذ" (الخطوات، الآليات، المخرجات، الحوكمة).

5) التعارض:
   - إذا تعارض USER_INPUTS مع RFP_EVIDENCE: قدّم RFP_EVIDENCE واذكر أن الشرط يتطلب مواءمة.

## متطلبات إلزامية داخل كل قسم:
- (أ) الأعمال/الأنشطة الرئيسية
- (ب) المخرجات المتوقعة (Deliverables) أو ما يقابلها
- (ج) آلية التنفيذ/الحوكمة/التواصل (حسب القسم)
- (د) بند "يتطلب تأكيد" للعناصر الناقصة (إن وجدت)

## المخرجات:
- اكتب العرض الفني كاملاً بصيغة Markdown ومنسقًا.
- لا تذكر أنك نموذج ذكاء اصطناعي ولا تذكر أدوات التوليد.
"""


def _clean(s: str) -> str:
    return (s or "").strip()


def generate_full_technical_offer(
    pair_id: int,
    project_title: str,
    scope_text: str = "",
    timeline_text: str = "",
    special_terms: str = "",
    top_k: int = 8,
    min_score: float = 0.30,
    temperature: float = 0.2,
    include_evidence_headers: bool = True,   # True للتطوير / False للإنتاج
) -> str:
    # 1) Build retrieval-based draft context (from your context_preparation function)
    draft_context = generate_tech_offer_draft_with_intents(
        pair_id=pair_id,
        project_title=project_title,
        structure=TECH_OFFER_STRUCTURE,
        client=qdrant,
        collection_name=COLLECTION,
        embed_fn=embed,
        intents=TECH_OFFER_INTENTS,
        include_tp_support=True,
        top_k=top_k,
        min_score=min_score,
        scope_text=_clean(scope_text),
        timeline_text=_clean(timeline_text),
        include_evidence_headers=include_evidence_headers
    )

    user_inputs = f"""
[USER_INPUTS]
- اسم المشروع: {_clean(project_title)}
- نطاق العمل (من المستخدم): {_clean(scope_text) if _clean(scope_text) else "غير محدد (يتطلب تأكيد)"}
- الجدول الزمني (من المستخدم): {_clean(timeline_text) if _clean(timeline_text) else "غير محدد (يتطلب تأكيد)"}
- شروط خاصة (من المستخدم): {_clean(special_terms) if _clean(special_terms) else "لا يوجد"}
"""

    prompt = f"""
{SYSTEM_PROMPT_FINAL}

{user_inputs}

[DRAFT_CONTEXT]
{draft_context}

## تعليمات تنفيذية أخيرة:
- اعتبر أي نص تحت "TP_STYLE" داخل السياق أسلوب فقط وليس مصدر حقائق.
- لا تكرر نصوص الأدلة كما هي؛ أعد صياغتها.
- إذا كان الدليل ضعيفًا في قسم ما: أضف فقرة قصيرة بعنوان "يتطلب تأكيد" توضح ما يلزم تأكيده.
- اكتب العرض الفني كاملاً بنفس العناوين وبنفس الترتيب، وبصياغة جديدة.
"""

    resp = llm.generate_content(
        prompt,
        generation_config={"temperature": temperature}
    )
    return (resp.text or "").strip()


# TEST: Generate Technical Offer


In [8]:
# ==============================
# TEST: Generate Technical Offer (STYLE ONLY)
# ==============================

from datetime import date
from qdrant_client.http import models as qm
import mdformat

SYSTEM_PROMPT_FINAL = """أنت خبير إعداد عروض فنية حكومية في المملكة العربية السعودية، ومتخصص في صياغة العروض الفنية وفق متطلبات الجهات الحكومية والأنظمة التنظيمية السعودية.

قواعد إلزامية:

1. اكتب بصيغة رسمية احترافية واضحة باللغة العربية.
2. اكتب محتوى تنفيذي يشرح "كيف سيتم التنفيذ" وليس وصفًا إنشائيًا عامًا.
3. التزم حرفيًا بعناوين وهيكل TECH_OFFER_STRUCTURE وبنفس الترتيب.
4. لا تخترع معلومات غير مذكورة في مدخلات المستخدم أو سياق RFP.
5. لا تنسخ أي نص حرفيًا من سياق TP_STYLE — استخدمه كمرجع أسلوبي فقط.
6. لا تذكر مصادر أو مراجع داخل النص النهائي.
7. إذا كانت هناك معلومات ناقصة، أنشئ فقرة بعنوان "يتطلب تأكيد" وحدد ما يحتاج توضيح.
8. استخدم Markdown منظم مع عناوين فرعية وقوائم نقطية وجداول عند الحاجة.
9. ركّز على القيمة المقدمة للجهة الحكومية والامتثال للمعايير السعودية.
10. اجعل العرض مترابطًا ومهنيًا وقابلًا للتنفيذ الفعلي.

مهم: الهدف هو إنتاج عرض فني جاهز للتقديم في منافسة حكومية سعودية.

"""

# ---------- Example External Inputs ----------
project_title = "مشروع تطوير منصة ذكية لإدارة طلبات الدعم الفني والخدمات الداخلية"

scope_text = """
- تحليل الوضع الحالي لآلية استقبال ومعالجة طلبات الدعم الفني والخدمات الداخلية.
- تصميم وتطوير منصة مركزية لإدارة الطلبات تشمل تسجيل الطلبات وتصنيفها آليًا.
- توجيه الطلبات للفرق المختصة بناءً على نوع الطلب وأولويته.
- متابعة حالة الطلبات ومؤشرات الأداء.
- توفير لوحة تحكم للإدارة لعرض التقارير والإحصاءات.
- التكامل مع الأنظمة الداخلية الحالية.
- إعداد وثائق الاستخدام والتشغيل وتدريب المستخدمين.
""".strip()

special_terms = """
- تشغيل المنصة داخل البنية التحتية الداخلية للجهة (On-Prem).
- الالتزام بسياسات الأمن السيبراني المعتمدة.
- عدم استخدام أي خدمات سحابية خارجية.
- تسليم الكود المصدري كاملًا مع حقوق الاستخدام.
- دعم اللغة العربية في واجهات النظام.
""".strip()

timeline_text = "مدة التنفيذ المقترحة 4 أشهر مقسمة على مراحل تحليل، تطوير، اختبار، وتسليم."

# ---------- Retrieve STYLE only (from all TP) ----------
def retrieve_tp_style(top_k: int = 10, max_chars_total: int = 3500):
    query = "أسلوب كتابة عرض فني حكومي مقدمة فهم نطاق العمل منهجية التنفيذ المخرجات الحوكمة"
    vec = embed(query)

    hits = qdrant.search(
        collection_name=COLLECTION,
        query_vector=vec,
        query_filter=qm.Filter(
            must=[
                qm.FieldCondition(key="doc_type", match=qm.MatchValue(value="tp"))
            ]
        ),
        limit=top_k,
        with_payload=True
    )

    style_blocks = []
    seen = set()
    total = 0

    for h in hits:
        txt = (h.payload.get("text", "") or "").strip()
        if not txt:
            continue

        # dedup by first 160 chars
        key = txt[:160]
        if key in seen:
            continue
        seen.add(key)

        chunk = txt[:450]  # قصير لتقليل tokens
        if total + len(chunk) > max_chars_total:
            break

        style_blocks.append(chunk)
        total += len(chunk)

    return "\n\n---\n\n".join(style_blocks)

TP_STYLE_CONTEXT = retrieve_tp_style(top_k=10, max_chars_total=3500)

# ---------- Prompt Assembly ----------
today_str = date.today().isoformat()
structure_text = "\n".join([f"- {s['title']}" for s in TECH_OFFER_STRUCTURE if s.get("required")])

prompt = f"""
{SYSTEM_PROMPT_FINAL}

[USER_INPUTS]
- اسم المشروع: {project_title}
- تاريخ التقديم: {today_str}

- نطاق العمل:
{scope_text}–

- الشروط الخاصة:
{special_terms}

- الجدول الزمني:
{timeline_text}

[TECH_OFFER_STRUCTURE]
{structure_text}

[TP_STYLE] (أسلوب فقط)
{TP_STYLE_CONTEXT}

قواعد إنتاج إلزامية:
- التزم حرفيًا بعناوين TECH_OFFER_STRUCTURE وبنفس ترتيبها.
- داخل كل قسم: اذكر (الأعمال/الأنشطة) + (المخرجات) + (آلية التنفيذ/الحوكمة) قدر الإمكان.
- لا تستخدم TP_STYLE كمصدر حقائق أو أرقام أو أسماء.
- إذا نقصت معلومة: ضع فقرة بعنوان "يتطلب تأكيد" وحدد ما يلزم تأكيده.

الآن: اكتب العرض الفني كاملًا بصيغة Markdown.
""".strip()

# ---------- Generate ----------
response = llm.generate_content(
    prompt,
    generation_config=genai.types.GenerationConfig(
        temperature=0.15,
        max_output_tokens=4096
    )
)

technical_offer_md = (response.text or "").strip()

# ---------- Optional: format ----------
technical_offer_md = mdformat.text(technical_offer_md)

# ---------- Preview ----------
print(technical_offer_md[:3500])
print("\n\n---\nLength (chars):", len(technical_offer_md))
print("Length (words):", len(technical_offer_md.split()))


D:\pip_cache\ipykernel_25952\3886101914.py:56: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = qdrant.search(


**عرض فني لمشروع تطوير منصة ذكية لإدارة طلبات الدعم الفني والخدمات الداخلية**

**تاريخ التقديم:** 2026-05-20

______________________________________________________________________

### 1) مقدمة وفهم عام للمشروع

تتشرف [اسم الجهة المقدمة للعرض - يتطلب تأكيد] بتقديم هذا العرض الفني لمشروع "تطوير منصة ذكية لإدارة طلبات الدعم الفني والخدمات الداخلية" للجهة الحكومية الموقرة. ندرك الأهمية الاستراتيجية لهذا المشروع في تعزيز الكفاءة التشغيلية، وتحسين جودة الخدمات الداخلية، وتسريع الاستجابة لطلبات الدعم الفني، بما يتماشى مع توجهات التحول الرقمي في المملكة العربية السعودية ورؤية 2030.

يهدف هذا العرض إلى توضيح فهمنا العميق لمتطلبات المشروع، وتقديم منهجية عمل متكاملة ومفصلة تضمن تنفيذًا ناجحًا وفعالًا للمنصة، مع الالتزام التام بالمعايير والأنظمة التنظيمية المعتمدة، لتقديم قيمة مضافة حقيقية للجهة.

______________________________________________________________________

### 2) فهم نطاق العمل والمتطلبات

لقد تم استيعاب نطاق العمل والمتطلبات الخاصة بالمشروع بشكل دقيق، والتي تشكل الأساس لتصميم وتنفيذ

In [9]:
print(technical_offer_md)


**عرض فني لمشروع تطوير منصة ذكية لإدارة طلبات الدعم الفني والخدمات الداخلية**

**تاريخ التقديم:** 2026-05-20

______________________________________________________________________

### 1) مقدمة وفهم عام للمشروع

تتشرف [اسم الجهة المقدمة للعرض - يتطلب تأكيد] بتقديم هذا العرض الفني لمشروع "تطوير منصة ذكية لإدارة طلبات الدعم الفني والخدمات الداخلية" للجهة الحكومية الموقرة. ندرك الأهمية الاستراتيجية لهذا المشروع في تعزيز الكفاءة التشغيلية، وتحسين جودة الخدمات الداخلية، وتسريع الاستجابة لطلبات الدعم الفني، بما يتماشى مع توجهات التحول الرقمي في المملكة العربية السعودية ورؤية 2030.

يهدف هذا العرض إلى توضيح فهمنا العميق لمتطلبات المشروع، وتقديم منهجية عمل متكاملة ومفصلة تضمن تنفيذًا ناجحًا وفعالًا للمنصة، مع الالتزام التام بالمعايير والأنظمة التنظيمية المعتمدة، لتقديم قيمة مضافة حقيقية للجهة.

______________________________________________________________________

### 2) فهم نطاق العمل والمتطلبات

لقد تم استيعاب نطاق العمل والمتطلبات الخاصة بالمشروع بشكل دقيق، والتي تشكل الأساس لتصميم وتنفيذ

In [10]:
len(technical_offer_md)

7106

In [11]:
# ==============================
# TEST: Generate Technical Offer (STYLE ONLY)
# ==============================

from datetime import date
from qdrant_client.http import models as qm
import mdformat

SYSTEM_PROMPT_FINAL = """أنت خبير إعداد عروض فنية حكومية في المملكة العربية السعودية، ومتخصص في صياغة العروض الفنية وفق متطلبات الجهات الحكومية والأنظمة التنظيمية السعودية.

قواعد إلزامية:

1. اكتب بصيغة رسمية احترافية واضحة باللغة العربية.
2. اكتب محتوى تنفيذي يشرح "كيف سيتم التنفيذ" وليس وصفًا إنشائيًا عامًا.
3. التزم حرفيًا بعناوين وهيكل TECH_OFFER_STRUCTURE وبنفس الترتيب.
4. لا تخترع معلومات غير مذكورة في مدخلات المستخدم أو سياق RFP.
5. لا تنسخ أي نص حرفيًا من سياق TP_STYLE — استخدمه كمرجع أسلوبي فقط.
6. لا تذكر مصادر أو مراجع داخل النص النهائي.
7. إذا كانت هناك معلومات ناقصة، أنشئ فقرة بعنوان "يتطلب تأكيد" وحدد ما يحتاج توضيح.
8. استخدم Markdown منظم مع عناوين فرعية وقوائم نقطية وجداول عند الحاجة.
9. ركّز على القيمة المقدمة للجهة الحكومية والامتثال للمعايير السعودية.
10. اجعل العرض مترابطًا ومهنيًا وقابلًا للتنفيذ الفعلي.

مهم: الهدف هو إنتاج عرض فني جاهز للتقديم في منافسة حكومية سعودية.

"""

# ---------- Example External Inputs (REPLACED) ----------
project_title = "مشروع بناء منصة موحدة لإدارة البيانات والتقارير المؤسسية (Enterprise Data Platform) مع حوكمة البيانات وتكاملات متعددة"

scope_text = """
- تقييم الوضع الحالي لمصادر البيانات لدى الجهة (قواعد بيانات تشغيلية، ملفات، نظم طرف ثالث).
- تحديد الأنظمة والخدمات الحرجة التي تتطلب استمرارية تشغيل عالية.
- تصميم وبناء منصة بيانات مؤسسية موحدة تشمل طبقات:
  1) Ingestion Layer: سحب البيانات من مصادر متعددة (Batch/Streaming) مع جدولة ومراقبة.
  2) Storage Layer: مستودع بيانات/بحيرة بيانات داخلية (On-Prem) مع طبقات Raw/Cleansed/Curated.
  3) Transformation Layer: بناء نماذج بيانات (Data Models) وقواعد جودة بيانات (Data Quality Rules).
  4) Serving Layer: إتاحة البيانات للتقارير والتحليلات ولوحات المعلومات عبر APIs أو أدوات BI.
- بناء كتالوج بيانات (Data Catalog) يتضمن:
  - تعريفات البيانات (Business Glossary)
  - نسب البيانات (Data Lineage)
  - مالكي البيانات (Data Owners) ومسؤوليها (Data Stewards)
- تطبيق إطار حوكمة بيانات كامل:
  - سياسات الوصول والتصنيف (سري/داخلي/عام)
  - إجراءات طلب الصلاحيات والموافقة
  - إدارة التغييرات على النماذج والتقارير
- تصميم نظام صلاحيات متعدد المستويات:
  - مستخدم أعمال / محلل / مدير / مسؤول منصة
  - صلاحيات على مستوى الجداول والحقول (Field-level) عند الحاجة
- تكاملات إلزامية (على الأقل 6 تكاملات):
  - نظام ERP
  - نظام الموارد البشرية
  - نظام الأصول/المشتريات
  - نظام البريد/الدليل النشط AD
  - نظام شكاوى أو خدمة عملاء
  - قواعد بيانات تشغيلية داخلية
- متطلبات الأداء:
  - دعم ما لا يقل عن 300 مستخدم داخلي.
  - زمن تحديث التقارير الحرجة كل 15 دقيقة (لبيانات محددة).
- بناء لوحة مؤشرات مؤسسية (Executive Dashboard) + 10 تقارير تشغيلية معيارية.
- إعداد وثائق تشغيل وصيانة شاملة + تدريب فني وتدريب مستخدمي الأعمال.
- تشغيل تجريبي (Pilot) لمدة 4 أسابيع مع قياس الجودة وتحسينات قبل الإطلاق النهائي.
""".strip()

special_terms = """
- تشغيل داخلي بالكامل (On-Prem) بدون أي خدمات سحابية عامة.
- الالتزام بإطار الأمن السيبراني المعتمد لدى الجهة ومعايير الهيئة الوطنية للأمن السيبراني (NCA) بقدر ما ينطبق.
- منع نقل أي بيانات حساسة خارج بيئة الجهة، ومنع استخدام أدوات مشاركة خارجية.
- تسجيل ومراقبة كاملة (Audit Logs) لكل عمليات الوصول للبيانات والتغييرات.
- متطلبات توفر الخدمة (SLA):
  - توافر 99.5% للمنصة خلال ساعات العمل
  - استجابة الأعطال الحرجة خلال 2 ساعة
- متطلبات النسخ الاحتياطي والتعافي:
  - RPO ≤ 30 دقيقة للبيانات الحرجة
  - RTO ≤ 4 ساعات للخدمات الأساسية
- تسليم الكود المصدري وملفات البنية وحقوق الاستخدام للجهة.
- دعم العربية بالكامل في التسميات والواجهات والتقارير.
- أي نقاط غير محددة يجب ذكرها تحت "يتطلب تأكيد" بدل التخمين.
""".strip()

timeline_text = "مدة التنفيذ 6 أشهر، تشمل: تحليل، تصميم، بناء، تكاملات، اختبار، تشغيل تجريبي، تسليم نهائي، وتدريب."

# ---------- Retrieve STYLE only (from all TP) ----------
def retrieve_tp_style(top_k: int = 10, max_chars_total: int = 3500):
    query = "أسلوب كتابة عرض فني حكومي مقدمة فهم نطاق العمل منهجية التنفيذ المخرجات الحوكمة"
    vec = embed(query)

    hits = qdrant.search(
        collection_name=COLLECTION,
        query_vector=vec,
        query_filter=qm.Filter(
            must=[
                qm.FieldCondition(key="doc_type", match=qm.MatchValue(value="tp"))
            ]
        ),
        limit=top_k,
        with_payload=True
    )

    style_blocks = []
    seen = set()
    total = 0

    for h in hits:
        txt = (h.payload.get("text", "") or "").strip()
        if not txt:
            continue

        # dedup by first 160 chars
        key = txt[:160]
        if key in seen:
            continue
        seen.add(key)

        chunk = txt[:450]  # قصير لتقليل tokens
        if total + len(chunk) > max_chars_total:
            break

        style_blocks.append(chunk)
        total += len(chunk)

    return "\n\n---\n\n".join(style_blocks)

TP_STYLE_CONTEXT = retrieve_tp_style(top_k=10, max_chars_total=3500)

# ---------- Prompt Assembly ----------
today_str = date.today().isoformat()
structure_text = "\n".join([f"- {s['title']}" for s in TECH_OFFER_STRUCTURE if s.get("required")])

prompt = f"""
{SYSTEM_PROMPT_FINAL}

[USER_INPUTS]
- اسم المشروع: {project_title}
- تاريخ التقديم: {today_str}

- نطاق العمل:
{scope_text}

- الشروط الخاصة:
{special_terms}

- الجدول الزمني:
{timeline_text}

[TECH_OFFER_STRUCTURE]
{structure_text}

[TP_STYLE] (أسلوب فقط)
{TP_STYLE_CONTEXT}

قواعد إنتاج إلزامية:
- التزم حرفيًا بعناوين TECH_OFFER_STRUCTURE وبنفس ترتيبها.
- داخل كل قسم: اذكر (الأعمال/الأنشطة) + (المخرجات) + (آلية التنفيذ/الحوكمة) قدر الإمكان.
- لا تستخدم TP_STYLE كمصدر حقائق أو أرقام أو أسماء.
- إذا نقصت معلومة: ضع فقرة بعنوان "يتطلب تأكيد" وحدد ما يلزم تأكيده.

الآن: اكتب العرض الفني كاملًا بصيغة Markdown.
""".strip()

# ---------- Generate ----------
response = llm.generate_content(
    prompt,
    generation_config=genai.types.GenerationConfig(
        temperature=0.15,
        max_output_tokens=7000
    )
)

technical_offer_md = (response.text or "").strip()

# ---------- Optional: format ----------
technical_offer_md = mdformat.text(technical_offer_md)

# ---------- Preview ----------
print(technical_offer_md[:3500])
print("\n\n---\nLength (chars):", len(technical_offer_md))
print("Length (words):", len(technical_offer_md.split()))


D:\pip_cache\ipykernel_25952\2005541507.py:88: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = qdrant.search(


## عرض فني لمشروع بناء منصة موحدة لإدارة البيانات والتقارير المؤسسية

**تاريخ التقديم:** 2026-05-20

______________________________________________________________________

### 1) مقدمة وفهم عام للمشروع

تتشرف [اسم الشركة المقدمة للعرض] بتقديم هذا العرض الفني لمشروع "بناء منصة موحدة لإدارة البيانات والتقارير المؤسسية (Enterprise Data Platform) مع حوكمة البيانات وتكاملات متعددة" للجهة الحكومية الموقرة. ندرك الأهمية الاستراتيجية لهذا المشروع في تمكين الجهة من اتخاذ قرارات مبنية على البيانات، وتحسين الكفاءة التشغيلية، وتعزيز الامتثال للمعايير الوطنية والدولية.

يأتي هذا المشروع في صميم التوجهات الوطنية نحو التحول الرقمي وحوكمة البيانات، بما يتماشى مع رؤية المملكة 2030. إن بناء منصة بيانات مؤسسية متكاملة ومؤمنة داخليًا (On-Premise) سيوفر للجهة بنية تحتية قوية لإدارة أصولها المعلوماتية بكفاءة عالية، مع ضمان أعلى مستويات الأمن السيبراني والخصوصية.

تلتزم [اسم الشركة المقدمة للعرض] بتقديم حلول تقنية مبتكرة وموثوقة، مع خبرة واسعة في تنفيذ مشاريع حكومية مماثلة في المملكة العربية السعودية، بما ي

In [12]:
# ==============================
# TEST: Generate Technical Offer (STYLE ONLY) - HARD EXAMPLE (MODIFIED)
# ==============================

from datetime import date
from qdrant_client.http import models as qm

# ---------- Example External Inputs (HARD / LONG OUTPUT) ----------
project_title = "مشروع بناء منصة موحدة لإدارة البيانات والتقارير المؤسسية (Enterprise Data Platform) مع حوكمة البيانات وتكاملات متعددة"

scope_text = """
- تقييم الوضع الحالي لمصادر البيانات لدى الجهة (قواعد بيانات تشغيلية، ملفات، نظم طرف ثالث).
- تحديد الأنظمة والخدمات الحرجة التي تتطلب استمرارية تشغيل عالية.
- تصميم وبناء منصة بيانات مؤسسية موحدة تشمل طبقات:
  1) Ingestion Layer: سحب البيانات من مصادر متعددة (Batch/Streaming) مع جدولة ومراقبة.
  2) Storage Layer: مستودع بيانات/بحيرة بيانات داخلية (On-Prem) مع طبقات Raw/Cleansed/Curated.
  3) Transformation Layer: بناء نماذج بيانات (Data Models) وقواعد جودة بيانات (Data Quality Rules).
  4) Serving Layer: إتاحة البيانات للتقارير والتحليلات ولوحات المعلومات عبر APIs أو أدوات BI.
- بناء كتالوج بيانات (Data Catalog) يتضمن:
  - تعريفات البيانات (Business Glossary)
  - نسب البيانات (Data Lineage)
  - مالكي البيانات (Data Owners) ومسؤوليها (Data Stewards)
- تطبيق إطار حوكمة بيانات كامل:
  - سياسات الوصول والتصنيف (سري/داخلي/عام)
  - إجراءات طلب الصلاحيات والموافقة
  - إدارة التغييرات على النماذج والتقارير
- تصميم نظام صلاحيات متعدد المستويات:
  - مستخدم أعمال / محلل / مدير / مسؤول منصة
  - صلاحيات على مستوى الجداول والحقول (Field-level) عند الحاجة
- تكاملات إلزامية (على الأقل 6 تكاملات):
  - نظام ERP
  - نظام الموارد البشرية
  - نظام الأصول/المشتريات
  - نظام البريد/الدليل النشط AD
  - نظام شكاوى أو خدمة عملاء
  - قواعد بيانات تشغيلية داخلية
- متطلبات الأداء:
  - دعم ما لا يقل عن 300 مستخدم داخلي.
  - زمن تحديث التقارير الحرجة كل 15 دقيقة (لبيانات محددة).
- بناء لوحة مؤشرات مؤسسية (Executive Dashboard) + 10 تقارير تشغيلية معيارية.
- إعداد وثائق تشغيل وصيانة شاملة + تدريب فني وتدريب مستخدمي الأعمال.
- تشغيل تجريبي (Pilot) لمدة 4 أسابيع مع قياس الجودة وتحسينات قبل الإطلاق النهائي.
"""

special_terms = """
- تشغيل داخلي بالكامل (On-Prem) بدون أي خدمات سحابية عامة.
- الالتزام بإطار الأمن السيبراني المعتمد لدى الجهة ومعايير الهيئة الوطنية للأمن السيبراني (NCA) بقدر ما ينطبق.
- منع نقل أي بيانات حساسة خارج بيئة الجهة، ومنع استخدام أدوات مشاركة خارجية.
- تسجيل ومراقبة كاملة (Audit Logs) لكل عمليات الوصول للبيانات والتغييرات.
- متطلبات توفر الخدمة (SLA):
  - توافر 99.5% للمنصة خلال ساعات العمل
  - استجابة الأعطال الحرجة خلال 2 ساعة
- متطلبات النسخ الاحتياطي والتعافي:
  - RPO ≤ 30 دقيقة للبيانات الحرجة
  - RTO ≤ 4 ساعات للخدمات الأساسية
- تسليم الكود المصدري وملفات البنية وحقوق الاستخدام للجهة.
- دعم العربية بالكامل في التسميات والواجهات والتقارير.
- أي نقاط غير محددة يجب ذكرها تحت "يتطلب تأكيد" بدل التخمين.
"""

timeline_text = "مدة التنفيذ 6 أشهر، تشمل: تحليل، تصميم، بناء، تكاملات، اختبار، تشغيل تجريبي، تسليم نهائي، وتدريب."

# ---------- Retrieve STYLE only (from all TP) ----------
def retrieve_tp_style(top_k: int = 18, max_chars_total: int = 9000):
    query = "أسلوب كتابة عرض فني حكومي مقدمة فهم نطاق العمل منهجية التنفيذ المخرجات الحوكمة"
    vec = embed(query)

    hits = qdrant.search(
        collection_name=COLLECTION,
        query_vector=vec,
        query_filter=qm.Filter(
            must=[
                qm.FieldCondition(
                    key="doc_type",
                    match=qm.MatchValue(value="tp")
                )
            ]
        ),
        limit=top_k,
        with_payload=True
    )

    style_blocks = []
    seen = set()
    total = 0

    for h in hits:
        txt = (h.payload.get("text", "") or "").strip()
        if not txt:
            continue

        key = txt[:160]
        if key in seen:
            continue
        seen.add(key)

        chunk = txt[:1200]
        if total + len(chunk) > max_chars_total:
            break

        style_blocks.append(chunk)
        total += len(chunk)

    return "\n\n---\n\n".join(style_blocks)


TP_STYLE_CONTEXT = retrieve_tp_style(top_k=18, max_chars_total=9000)

# ---------- Prompt Assembly ----------
today_str = date.today().isoformat()

prompt = f"""
{SYSTEM_PROMPT_FINAL}

[USER_INPUTS]
- اسم المشروع: {project_title}
- تاريخ التقديم: {today_str}
- نطاق العمل:
{scope_text}

- الشروط الخاصة:
{special_terms}

- الجدول الزمني:
{timeline_text}

[TECH_OFFER_STRUCTURE]
{TECH_OFFER_STRUCTURE}

[TP_STYLE]  (أسلوب فقط)
{TP_STYLE_CONTEXT}

قواعد إنتاج إلزامية:
- التزم حرفيًا بعناوين TECH_OFFER_STRUCTURE وبنفس ترتيبها.
- اكتب محتوى تنفيذي (كيف ننفّذ) وليس إنشائي.
- داخل كل قسم: اذكر (الأعمال/الأنشطة) + (المخرجات) + (آلية التنفيذ/الحوكمة) قدر الإمكان.
- لا تستخدم TP_STYLE كمصدر حقائق أو أرقام أو أسماء.
- إذا نقصت معلومة: ضع فقرة بعنوان "يتطلب تأكيد" وحدد ما يلزم تأكيده.

الآن: اكتب العرض الفني كاملًا بصيغة Markdown.
""".strip()

# ---------- Generate (MODIFIED) ----------
response = llm.generate_content(
    prompt,
    generation_config=genai.types.GenerationConfig(
        temperature=0.15,
        max_output_tokens=7000,   
        candidate_count=1    
    )
)

technical_offer_md = (response.text or "").strip()

# ---------- Preview ----------
print(technical_offer_md[:3500])
print("\n\n---\nLength (chars):", len(technical_offer_md))
print("Length (words):", len(technical_offer_md.split()))


D:\pip_cache\ipykernel_25952\2534767905.py:68: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = qdrant.search(


تاريخ التقديم: 2026-05-20

---

# العرض الفني لمشروع بناء منصة موحدة لإدارة البيانات والتقارير المؤسسية

## 1) مقدمة وفهم عام للمشروع

تتشرف [اسم الجهة المقدمة للعرض - يتطلب تأكيد] بتقديم هذا العرض الفني لمشروع "بناء منصة موحدة لإدارة البيانات والتقارير المؤسسية (Enterprise Data Platform) مع حوكمة البيانات وتكاملات متعددة" للجهة الحكومية الموقرة. ندرك الأهمية الاستراتيجية لهذا المشروع في تمكين الجهة من استغلال أصولها البياناتية بفعالية، وتحويلها إلى رؤى قابلة للتنفيذ تدعم اتخاذ القرار، وتعزز الكفاءة التشغيلية، وتضمن الامتثال للمتطلبات التنظيمية.

يهدف هذا العرض إلى توضيح فهمنا العميق لمتطلبات المشروع، وتقديم منهجية تنفيذية واضحة ومفصلة تضمن تحقيق الأهداف المرجوة بكفاءة وجودة عالية، مع الالتزام التام بالمعايير والأنظمة الحكومية السعودية، خاصة فيما يتعلق بالأمن السيبراني وحوكمة البيانات. سنعمل على بناء منصة بيانات مؤسسية متكاملة، قابلة للتوسع، ومصممة خصيصًا للعمل ضمن البيئة الداخلية للجهة (On-Premise)، لضمان أقصى درجات التحكم والأمان.

## 2) فهم نطاق العمل والمتطلبات

لقد قمنا بتحليل شام

In [13]:
print(technical_offer_md)
len(technical_offer_md)

تاريخ التقديم: 2026-05-20

---

# العرض الفني لمشروع بناء منصة موحدة لإدارة البيانات والتقارير المؤسسية

## 1) مقدمة وفهم عام للمشروع

تتشرف [اسم الجهة المقدمة للعرض - يتطلب تأكيد] بتقديم هذا العرض الفني لمشروع "بناء منصة موحدة لإدارة البيانات والتقارير المؤسسية (Enterprise Data Platform) مع حوكمة البيانات وتكاملات متعددة" للجهة الحكومية الموقرة. ندرك الأهمية الاستراتيجية لهذا المشروع في تمكين الجهة من استغلال أصولها البياناتية بفعالية، وتحويلها إلى رؤى قابلة للتنفيذ تدعم اتخاذ القرار، وتعزز الكفاءة التشغيلية، وتضمن الامتثال للمتطلبات التنظيمية.

يهدف هذا العرض إلى توضيح فهمنا العميق لمتطلبات المشروع، وتقديم منهجية تنفيذية واضحة ومفصلة تضمن تحقيق الأهداف المرجوة بكفاءة وجودة عالية، مع الالتزام التام بالمعايير والأنظمة الحكومية السعودية، خاصة فيما يتعلق بالأمن السيبراني وحوكمة البيانات. سنعمل على بناء منصة بيانات مؤسسية متكاملة، قابلة للتوسع، ومصممة خصيصًا للعمل ضمن البيئة الداخلية للجهة (On-Premise)، لضمان أقصى درجات التحكم والأمان.

## 2) فهم نطاق العمل والمتطلبات

لقد قمنا بتحليل شام

11826

In [14]:
len(technical_offer_md)


11826

In [15]:
print(llm.model_name)


models/gemini-2.5-flash


In [16]:
# ==============================
# TEST: Generate Technical Offer (STYLE ONLY) - HARD EXAMPLE (MODIFIED)
# ==============================

from datetime import date
from qdrant_client.http import models as qm

# ---------- Example External Inputs (HARD / LONG OUTPUT) ----------
project_title = " مشروع إدارة وتشغيل وصيانة منظومة تقنية المعلومات والتحول الرقمي للأمن العام (اتفاق مباشر بين الجهات الحكومية استناداً للمادة 89 من النظام) مشروع إدارة وتشغيل وصيانة منظومة تقنية المعلومات والتحول الرقمي للأمن العام (اتفاق مباشر بين الجهات الحكومية استناداً للمادة 89 من النظام)"

scope_text = """
نطاق عمل المشروع:
يشمل نطاق عمل المشروع الإدارة المتكاملة للخدمات التقنية المقدمة في الأمن العام بما يكفل تنفيذ المخرجات المطلوبة والحفاظ على بنية تقنية المعلومات وجميع الأعمال الإلكترونية بعموم ما يشمله النطاق المكاني مع القيام بما يلزم من أعمال الإدارة والتشغيل والصيانة والتحديث والتهيئة والفك والنقل وإعادة التركيب اللازم لها وتوفير قطع الغيار، بما يضمن صلاحيتها للعمل والتشغيل والاستخدام المستمر طوال (24) ساعة يوميا، بما فيها ايام الجمعة والسبت والعطلات الرسمية. والقيام كذلك بإصلاح جميع الأعطال على الأجهزة والبرامج التشغيلية وذلك طبقا لأرقى المستويات الفنية، مع تأمين كافة ما يلزم لأعمال إدارة الخدمات، وبما يتوافق مع الأنظمة والتشريعات الحكومية ذات العلاقة، وتوفير التقارير المطلوبة لضمان سير العمل على أن تشمل الخدمات المدارة على ما يلي:
•	الخدمات المدارة للبنية المؤسسية
•	الخدمة المدارة للتحول الرقمي
•	الخدمة المدارة لإدارة المشاريع
•	الخدمة المدارة للنظم والبرامج
•	الخدمة المدارة للبنية التحتية ومراكز البيانات
•	الخدمة المدارة للبيانات والذكاء الاصطناعي
•	الخدمات المدارة لهيكلية البنية التحتية
•	توفير قطع الغيار 

"""

special_terms = """
1.	يلتزم المتعاقد بالفصل الواضح بين مخرجات مسارات الخدمات المدارة وفق ما هو موضح في نطاق العمل، دون دمج أو تداخل في المسؤوليات أو الأدوار.
2.	يلتزم المتعاقد باستمرارية الرخص التقنية – متى وردت ضمن نطاق العمل – باعتبارها خدمات تشغيلية، وذلك دون الإخلال بأي التزامات مالية أو تعاقدية واردة في العقد.
3.	يلتزم المتعاقد بتأمين قطع الغيار اللازمة – إن وجدت – والالتزام باتفاقيات مستوى الخدمة (SLA) المعتمدة.
4.	يلتزم المتعاقد بتقديم خدمات الدعم الفني وفق ما ورد في ملحق (6)، وبما يضمن استمرارية الخدمات والالتزام بمستويات الخدمة المعتمدة، مع توثيق البلاغات ورفع التقارير الدورية بشأنها.
5.	يقر المتعاقد باطلاعه الكامل على نطاق العمل وجميع ملاحقه، وفهمه لطبيعة الأعمال المطلوبة، ويتحمل كامل المسؤولية عن تنفيذها وفق ما ورد في الكراسة دون الرجوع على الأمن العام بأي مطالبات ناتجة عن سوء الفهم أو التقدير.
6.	يلتزم المتعاقد بتنفيذ جميع الأعمال والمخرجات وفق الأنظمة والتشريعات الحكومية المعمول بها في المملكة العربية السعودية، وبما يشمل أنظمة المنافسات والمشتريات الحكومية واللوائح والتعليمات ذات العلاقة.
7.	تُعد التقارير الدورية والمرحلية والنهائية جزءًا من المخرجات المطلوبة ضمن نطاق العمل، ويلتزم المتعاقد بتقديمها وفق النماذج والآليات المعتمدة لدى الأمن العام، وبالجودة والمواعيد المحددة، ويخضع تقييمها لمرئيات الجهة المشرفة.
8.	يخضع أداء المتعاقد للتقييم الدوري من قبل الأمن العام وفق مؤشرات الأداء واتفاقيات مستوى الخدمة المعتمدة، ويترتب على نتائج التقييم تطبيق ما ورد في العقد من إجراءات تصحيحية أو غرامات أو أي تدابير نظامية أخرى.
9.	يلتزم المتعاقد بتوفير الكوادر المؤهلة لتنفيذ نطاق العمل، والحصول على موافقة الأمن العام على العناصر الرئيسية، مع الالتزام باستبدال أي عنصر غير مناسب خلال مدة تحددها الجهة دون أي تكلفة إضافية.
10.	تكون جميع المخرجات والوثائق والبيانات والنماذج الناتجة عن تنفيذ العقد، سواء كانت ورقية أو إلكترونية، ملكًا للأمن العام، ولا يجوز للمتعاقد استخدامها أو التصرف بها إلا بموافقة خطية مسبقة من الأمن العام.
11.	يلتزم المتعاقد بالتعاون الكامل مع فرق الأمن العام والجهات ذات العلاقة، وتقديم الدعم اللازم لضمان استمرارية الأعمال وعدم تعطل الخدمات، مع الالتزام بتطبيق خطط الطوارئ المعتمدة عند الحاجة.
12.	لا يخل ما ورد في هذه الشروط الخاصة بحق الأمن العام في اتخاذ أي إجراء نظامي أو تعاقدي منصوص عليه في العقد أو الأنظمة ذات العلاقة، متى ما أخل المتعاقد بالتزاماته.


"""

timeline_text = "مدة التنفيذ 12 اشهر."

# ---------- Retrieve STYLE only (from all TP) ----------
def retrieve_tp_style(top_k: int = 18, max_chars_total: int = 9000):
    query = "أسلوب كتابة عرض فني حكومي مقدمة فهم نطاق العمل منهجية التنفيذ المخرجات الحوكمة"
    vec = embed(query)

    hits = qdrant.search(
        collection_name=COLLECTION,
        query_vector=vec,
        query_filter=qm.Filter(
            must=[
                qm.FieldCondition(
                    key="doc_type",
                    match=qm.MatchValue(value="tp")
                )
            ]
        ),
        limit=top_k,
        with_payload=True
    )

    style_blocks = []
    seen = set()
    total = 0

    for h in hits:
        txt = (h.payload.get("text", "") or "").strip()
        if not txt:
            continue

        key = txt[:160]
        if key in seen:
            continue
        seen.add(key)

        chunk = txt[:1200]
        if total + len(chunk) > max_chars_total:
            break

        style_blocks.append(chunk)
        total += len(chunk)

    return "\n\n---\n\n".join(style_blocks)


TP_STYLE_CONTEXT = retrieve_tp_style(top_k=18, max_chars_total=9000)

# ---------- Prompt Assembly ----------
today_str = date.today().isoformat()

prompt = f"""
{SYSTEM_PROMPT_FINAL}

[USER_INPUTS]
- اسم المشروع: {project_title}
- تاريخ التقديم: {today_str}
- نطاق العمل:
{scope_text}

- الشروط الخاصة:
{special_terms}

- الجدول الزمني:
{timeline_text}

[TECH_OFFER_STRUCTURE]
{TECH_OFFER_STRUCTURE}

[TP_STYLE]  (أسلوب فقط)
{TP_STYLE_CONTEXT}

قواعد إنتاج إلزامية:
- التزم حرفيًا بعناوين TECH_OFFER_STRUCTURE وبنفس ترتيبها.
- اكتب محتوى تنفيذي (كيف ننفّذ) وليس إنشائي.
- داخل كل قسم: اذكر (الأعمال/الأنشطة) + (المخرجات) + (آلية التنفيذ/الحوكمة) قدر الإمكان.
- لا تستخدم TP_STYLE كمصدر حقائق أو أرقام أو أسماء.
- إذا نقصت معلومة: ضع فقرة بعنوان "يتطلب تأكيد" وحدد ما يلزم تأكيده.

الآن: اكتب العرض الفني كاملًا بصيغة Markdown.
""".strip()

# ---------- Generate (MODIFIED) ----------
response = llm.generate_content(
    prompt,
    generation_config=genai.types.GenerationConfig(
        temperature=0.15,
        max_output_tokens=7000,   
        candidate_count=1    
    )
)

technical_offer_md = (response.text or "").strip()

# ---------- Preview ----------
print(technical_offer_md[:3500])
print("\n\n---\nLength (chars):", len(technical_offer_md))
print("Length (words):", len(technical_offer_md.split()))


D:\pip_cache\ipykernel_25952\2290784775.py:49: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = qdrant.search(


# العرض الفني لمشروع إدارة وتشغيل وصيانة منظومة تقنية المعلومات والتحول الرقمي للأمن العام

**تاريخ التقديم:** 2026-05-20

---

## 1) مقدمة وفهم عام للمشروع

تتشرف [اسم الجهة المقدمة للعرض] بتقديم هذا العرض الفني لمشروع إدارة وتشغيل وصيانة منظومة تقنية المعلومات والتحول الرقمي للأمن العام، والذي يأتي في إطار اتفاق مباشر بين الجهات الحكومية استناداً للمادة 89 من نظام المنافسات والمشتريات الحكومية. ندرك الأهمية الاستراتيجية لهذا المشروع في دعم رسالة الأمن العام وتعزيز قدراته التشغيلية والتحولية عبر توفير بيئة تقنية مستقرة، آمنة، وفعالة.

يتجلى فهمنا للمشروع في كونه ركيزة أساسية لضمان استمرارية وكفاءة جميع الخدمات التقنية والأعمال الإلكترونية الحيوية للأمن العام. نلتزم بتقديم حلول متكاملة تضمن أعلى مستويات الجودة والامتثال للمعايير الوطنية والدولية، بما يسهم في تحقيق الأهداف التشغيلية والتطويرية للأمن العام ويعزز مسيرته نحو التحول الرقمي الشامل.

---

## 2) فهم نطاق العمل والمتطلبات

لقد تم استيعاب نطاق عمل المشروع بشكل كامل، والذي يرتكز على الإدارة المتكاملة للخدمات التقنية في الأمن العا

In [17]:
print(technical_offer_md)
len(technical_offer_md)

# العرض الفني لمشروع إدارة وتشغيل وصيانة منظومة تقنية المعلومات والتحول الرقمي للأمن العام

**تاريخ التقديم:** 2026-05-20

---

## 1) مقدمة وفهم عام للمشروع

تتشرف [اسم الجهة المقدمة للعرض] بتقديم هذا العرض الفني لمشروع إدارة وتشغيل وصيانة منظومة تقنية المعلومات والتحول الرقمي للأمن العام، والذي يأتي في إطار اتفاق مباشر بين الجهات الحكومية استناداً للمادة 89 من نظام المنافسات والمشتريات الحكومية. ندرك الأهمية الاستراتيجية لهذا المشروع في دعم رسالة الأمن العام وتعزيز قدراته التشغيلية والتحولية عبر توفير بيئة تقنية مستقرة، آمنة، وفعالة.

يتجلى فهمنا للمشروع في كونه ركيزة أساسية لضمان استمرارية وكفاءة جميع الخدمات التقنية والأعمال الإلكترونية الحيوية للأمن العام. نلتزم بتقديم حلول متكاملة تضمن أعلى مستويات الجودة والامتثال للمعايير الوطنية والدولية، بما يسهم في تحقيق الأهداف التشغيلية والتطويرية للأمن العام ويعزز مسيرته نحو التحول الرقمي الشامل.

---

## 2) فهم نطاق العمل والمتطلبات

لقد تم استيعاب نطاق عمل المشروع بشكل كامل، والذي يرتكز على الإدارة المتكاملة للخدمات التقنية في الأمن العا

13574

## Preflight Check

This script verifies the generation pipeline before calling Gemini.

It confirms:
- Required TECH_OFFER_STRUCTURE sections are loaded
- SYSTEM_PROMPT_FINAL is present
- Qdrant returns relevant chunks

If Qdrant returns results and structure appears, the system is ready for generation.


In [18]:
from qdrant_client.http import models as qm


def preflight_check(project_title, pair_id=None, doc_type="tp", top_k=5):
    # 1. Show required structure
    print("=== STRUCTURE (required sections) ===")
    for s in TECH_OFFER_STRUCTURE:
        if s.get("required"):
            print("-", s["title"])

    # 2. Show system prompt preview
    print("\n=== SYSTEM PROMPT PREVIEW ===")
    print(SYSTEM_PROMPT_FINAL[:300])

    # 3. Build search query
    query = f"{project_title} technical proposal methodology scope deliverables"
    vec = embed(query)

    # 4. Search Qdrant by doc_type only بدون pair_id
    conditions = [
        qm.FieldCondition(
            key="doc_type",
            match=qm.MatchValue(value=doc_type),
        )
    ]

    hits = qdrant.search(
        collection_name=COLLECTION,
        query_vector=vec,
        query_filter=qm.Filter(must=conditions),
        limit=top_k,
        with_payload=True,
    )

    # 5. Print Qdrant results
    print(f"\n=== QDRANT RESULTS ({len(hits)} hits) ===")
    for i, h in enumerate(hits, 1):
        txt = (h.payload.get("text", "") or "").replace("\n", " ")
        print(f"\nHit {i} | score={h.score:.3f}")
        print(txt[:200])

    print("\nPreflight check complete. No Gemini call executed.")


# Example run
preflight_check(
    project_title="Enterprise Data Platform with Data Governance",
    pair_id=None,
    doc_type="tp",
)

=== STRUCTURE (required sections) ===
- 1) مقدمة وفهم عام للمشروع
- 2) فهم نطاق العمل والمتطلبات
- 3) منهجية التنفيذ وآلية العمل
- 4) خطة العمل والجدول الزمني
- 5) المخرجات والتسليمات
- 10) الالتزام والمتطلبات النظامية

=== SYSTEM PROMPT PREVIEW ===
أنت خبير إعداد عروض فنية حكومية في المملكة العربية السعودية، ومتخصص في صياغة العروض الفنية وفق متطلبات الجهات الحكومية والأنظمة التنظيمية السعودية.

قواعد إلزامية:

1. اكتب بصيغة رسمية احترافية واضحة باللغة العربية.
2. اكتب محتوى تنفيذي يشرح "كيف سيتم التنفيذ" وليس وصفًا إنشائيًا عامًا.
3. التزم حرف

=== QDRANT RESULTS (5 hits) ===

Hit 1 | score=0.593
حصر وتقييم الموردين المتاحين وحلول الانظمه. تحديد نطاق عمل النظام والمتطلبات الوظيفيه والفنيه. تطوير وثيقه مقارنه بين الحلول التقنيه المقترحه وتحديد الانسب لاتمته العمليات. المخرجات الرييسه: النموذج ا

Hit 2 | score=0.591
تحديد الاهداف والغرض من عمليه الارشفه الالكترونيه والمعلومات المراد ارشفتها، ثم تشكيل فرق عمل متخصص في الارشفه الالكترونيه. تصميم النظام: اختيار النظام او البرمجيات المناسبه

D:\pip_cache\ipykernel_25952\3555694021.py:27: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = qdrant.search(
